# Importing microbiome data

**R equivalent:** `import_biom()`, `import_qiime()`, `import_mothur()`, manual `phyloseq()` construction

This notebook shows how to load data from every format phyla supports and
construct a `Phyloseq` object.

In [ ]:
import pandas as pd
import numpy as np
import phyla
from phyla import (
    Phyloseq, OtuTable, SampleData, TaxTable, PhyTree,
    read_biom, read_qiime, read_mothur, read_csv,
)
from phyla.testing.fixtures import (
    load_global_patterns_reference,
    load_enterotype_reference,
    load_esophagus_reference,
)

print(f"phyla {phyla.__version__}")

## Manual construction from DataFrames

The most flexible approach — useful when data is already in Python.

In [ ]:
# OTU table: taxa as rows, samples as columns
otu_data = pd.DataFrame(
    {"Sample1": [100, 50, 0, 20],
     "Sample2": [80, 30, 10, 0],
     "Sample3": [0, 60, 40, 5]},
    index=["OTU1", "OTU2", "OTU3", "OTU4"],
)
otu = OtuTable(otu_data, taxa_are_rows=True)

# Sample metadata
sam = SampleData(pd.DataFrame(
    {"SampleType": ["Gut", "Skin", "Oral"],
     "Subject":   ["A",   "A",    "B"]},
    index=["Sample1", "Sample2", "Sample3"],
))

# Taxonomic classification
tax = TaxTable(pd.DataFrame(
    {"Phylum":  ["Firmicutes", "Bacteroidetes", "Proteobacteria", "Firmicutes"],
     "Genus":   ["Lactobacillus", "Bacteroides", "Pseudomonas", "Clostridium"]},
    index=["OTU1", "OTU2", "OTU3", "OTU4"],
))

ps = Phyloseq(otu=otu, sam=sam, tax=tax)
print(ps)

## From the golden reference datasets

These mirror R's `data("GlobalPatterns")` etc.

In [ ]:
ref = load_global_patterns_reference()
gp = Phyloseq(
    otu=OtuTable(ref["otu_table"], taxa_are_rows=True),
    sam=SampleData(ref["sample_data"]),
    tax=TaxTable(ref["tax_table"]),
    tree=PhyTree.from_newick(ref["phy_tree_newick"]),
)
print(gp)
print(f"\ntaxa: {gp.ntaxa}, samples: {gp.nsamples}")

In [ ]:
# R reference: ntaxa(GlobalPatterns) == 19216, nsamples == 26
assert gp.ntaxa == 19216, f"Expected 19216 taxa, got {gp.ntaxa}"
assert gp.nsamples == 26, f"Expected 26 samples, got {gp.nsamples}"
print("✓ GlobalPatterns dimensions match R reference")

## Inspect the object

**R equivalent:** `taxa_names()`, `sample_names()`, `rank_names()`, `sample_variables()`

In [ ]:
print("Rank names:      ", gp.rank_names)
print("Sample variables:", gp.sample_variables)
print("Sample names[:3]:", list(gp.sample_names[:3]))
print("Taxa names[:3]:  ", list(gp.taxa_names[:3]))